## Red Black Tree

### TL;DR

- We've already covered B Trees in the previous set of notes

- A Red Black Tree is literally a B Tree with branching factor of 4, except instead of storing keys and children within the same node, we use "colour" to group nodes that are supposed to be "together" while retaining a BST structure
    - So rather than allocating large, variable-sized array nodes to store multiple keys and children together, an RBT uses standard Binary Search Tree (BST) nodes and a single extra bit of data ("Color") to group keys conceptually into virtual B-Tree nodes.

- In all seriousness, this is all you need to keep in mind. Every arbitrary "Rule" that a red black tree has, is ultimately the result of trying to replicate a B-Tree structure without creating a B Tree
    - Every arbitrary "rule", rotation, or recoloring in an RBT is simply a way to replicate B-Tree operations without the structural memory overhead

### Why do we even need Red Black Trees to "replicate" B Trees?

- Before we get into the detail of **how** a Red Black Tree replicates B Trees, we should really ask why we even care about this. This section isn't really necessary, so skip it if you are in a rush. 

- Obviouly, this seems like a lot of unnecessary work. Why do we use Red Black Trees to replicate a B Tree of degree 4 when we can just...use a B tree?

- To understand why, we need to know how hardware architecture affects these data structures in two main environments: Disk and RAM.

- Disk
    - Disk reads are slow, because accessing data typically requires a physical or controller-level process to locate a large block of memory (usually around 4KB).
    - Getting to the right block takes a long time relative to operations performed on the data once it's loaded.
    - Therefore, when accessing data on Disk, we try our best to maximize the amount of information per block. B-Trees use massive nodes to minimize the number of disk lookups needed.
    - What happens if we try to use an RBT in a disk environment? 
        - For 1 billion items, a balanced binary tree is $\approx 30$ levels deep ($\log_2(1,000,000,000)$)
        - Because nodes are scattered randomly on disk, a search requires up to 30 slow disk reads
        - Furthermore, fetching a tiny 32-byte RBT node forces a 4KB block read anyway, wasting 99% of the I/O capacity
    - What happens if we try to use a B-Tree (say, with 300 branching factor) in a disk environment?
        - For 1 billion items, a B-Tree is $\approx 4$ levels deep ($\log_{300}(1,000,000,000)$)
        - Nodes are still scattered randomly on disk, but we only need to search the disk 4 times
        - We maximise information received with every disk fetch, because a B-Tree node takes up the full 4KB block
    - **Conclusion:** On a disk environment, B-Trees always dominate

- RAM
    - Comparing B-Trees and RBTs on RAM is much less straightforward. 
    
    - Let's first consider how CPU performs memory access with RAM
        - RAM does not return data in huge 4KB blocks. Instead, RAM fetches data in tiny 64-byte chunks called cache lines. 
        - In modern systems, this data fetching is actually pretty smart; the hardware reader comprises both a reader and a **prefetcher**
        - If you access some cache line, and the next cache line is some fixed distance away, the prefetcher now knows that you're reading a contiguous block and memory, and will pre-fetch the adjacent objects to store in L2 and L3 cache
    
    - How does this affect B-Trees vs RBTs?
        - Since B-Tree nodes store contiguous arrays, even though the node doesn't fit into a 64 byte cache line, prefetcher ensures that when you access a node, the entire array's access is **extremely fast**. Like sub-nanosecond fast!
        - RBTs, however, do not benefit from prefetching. This is because RBT nodes are created at different times, and memory is only allocated when the nodes are created
            - This is unlike the B-Tree case, the entire array is allocated when the B Tree is created
            - This means that the prefetcher doesn't work for RBTs, because there is no consistent access pattern! Nodes can be arbitrarily near or far from each other depending on when the allocation occurs. 
            - So to access a node, you have no choice but to follow an entire chain of pointers, which makes it less performant because every pointer call needs to load a new cache line.
        - Therefore, prefetching in RAM benefits B-Trees much more than RBTs!!

    - Nonetheless, for most in-memory self balancing trees IRL, we actually use RBTs! Why?
        - In working with self balancing trees, there are 2 expensive operations; (i) getting the right node, (ii) updating the tree
        - Prefetching solves (i) 
        - But it doesn't change the fact that a B-Tree is much more complicated to update and wasteful to create than an RBT!

    - Let's examine why:
        - Memory Allocation:
            - B-Tree nodes are large, and the entire array must be allocated when the node is first created. However, if a B-Tree node is not fully utilised, this operation is expensive at scale
            - This is especially expensive for RAM, which is a highly contested shared system resource
            - An RBT, on the other hand, allocates memory dynamically and strictly on-demand. It instantiates exactly one tiny node per element. There is zero structural padding, zero pre-allocated array overhead, and zero wasted RAM space.
        - Easier modifications:
            - Since a B-Tree node must always be ordered, insertion will cause data to be rearranged. When data overflows the node size, we need to perform a "node split", which allocates an entire new array and results in a copy of the existing array. These are all expensive operations
            - An RBT never shifts data or copies arrays, because it is simply a BST with pointers. Inserting or deleting an item requires creating 1 node anywhere in RAM and re-linking 2 or 3 pointers, which minimise copying and shuffling
        - Pointer Stability:
            - Since elements in a B-Tree array are constantly shifting, the absolute memory address of an element changes constantly. This breaks direct memory pointers, references, and iterators, which is hard to manage
            - Once a node is created in an RBT, its physical memory address never changes until it is completely deleted from the tree. The node can be recolored or its pointers re-routed during a rotation, but it stays anchored in the exact same spot in RAM. This guarantees absolute pointer stability, a foundational requirement for robust software design.
    - **Conclusion:** In a RAM environment, B Trees are faster to lookup since they exploit prefetching, BUT much more difficult to manage. Hence, RBTs generally prefferred.

- TLDR; B Trees and Red Black trees both have their place and time to shine. So it's important to learn and understand both!

### 5 rules to make Red Black Trees Work

- As we mentioned above, Red Black Trees are literally B Trees with branching factor of 4 but in disguise

- "Color" in an RBT simply serves as a way to demarcate the "start" of a new node in a B Tree. 
    - Black Node: Represents a "true" standalone parent node and marks a level boundary change in the B-Tree layout.
    - Red Node: Represents a key that conceptually belongs to the same B-Tree node as its black parent. It is a "co-inhabitant" of that larger block, pushed down into a binary format to avoid allocating an array.

- When we build a Red-Black tree, we must ensure the following **5 rules** to preserve this B-Tree structure:
    - Every node is either Red or Black
        - This is the basic tracking mechanism 
        - Every node needs one extra bit of data to tell the system whether it represents a standalone B-Tree node (Black) or a sub-component of a larger B-Tree node (Red)
    - The Root is always Black
        - By definition, since a black node represents a boundary in a B Tree, we must always start with a black node
    - Every leaf node (NIL/Null pointer) is Black
        - This helps us make the resolution of tree height consistent, and avoids the need to check if the last node is leaf or not a leaf
        - Usually it is implemented as a singleton black NULL node
    - If a node is Red, both its children must be Black.
        - No "Double Reds". 
        - Enforces that a virtual B-Tree node cannot contain more than 3 keys, preventing it from exceeding a branching factor of 4
    - Every path from a given node to any of its descendant NIL leaves contains the same number of Black nodes
        - B-Trees are always perfectly balanced. This enforces the balance behaviour


### Operations in managing RBTs

- There are 2 major patterns to understand when working with RBTs, especially in view of the 5 rules above; **rotations** and **insertions**

#### Tree Rotations

- This is actually nothing new, we have already done this extensively in the section on AVL trees

- A rotation is a restructuring of a BST that changes the parent-child heights *without* breaking the fundamental binary search invariant (Left < Parent <= Right).

- Rotations are used to shift a deeper subtree upward and push a shorter subtree downward.

- Left Rotation
    - When we left-rotate around a node `X`, we are making its right child `Y` the new parent of the subtree.
    - **The Pivot:** `X` moves down to become the left child of `Y`.
    - **The Orphan Hand-off:** `Y`'s original left child (`beta`) is handed off to become `X`'s new right child.
        ```
        X (Old Root)                       Y (New Root)
        / \                                / \
        alpha Y                            X   gamma
            / \                          / \
        beta  gamma                  alpha beta
        ```

- Right Rotation
    - A right rotation is the exact mirror image of a left rotation. When we right-rotate around a node `Y`:

    - **The Pivot:** `Y` moves down to become the right child of `X`.
    - **The Orphan Hand-off:** `X`'s original right child (`beta`) is handed off to become `Y`'s new left child.

        ```       
            Y (Old Root)                     X (New Root)
            / \                              / \
            X   gamma                    alpha   Y
            / \                                  / \
        alpha beta                            beta  gamma
        ```

#### Tree Mutations: Insertion Rules

- When a new item is inserted into a Red-Black Tree, we always insert it as a **RED node**. 
    - Why? Because inserting a Red node guarantees we do not violate **Rule 5 (Perfect Black Balance)**. 
        - No paths gain an extra black node, so the baseline B-Tree height remains unchanged.
    - But inserting a Red node right next to an existing Red parent can cause **Double Red Violation (Rule 4)**. 
    - To resolve a Double Red violation, we will look at the color of the newly inserted node's **Uncle** (parent's sibling) 
    - Let's go through the possible cases:

- Case 1: The Uncle is RED (Recoloring Only)
    - This scenario represents a structural overflow in our virtual B-Tree. 
    - Why? 
        - Because if the uncle is red, and the parent is red, the parent + uncle + parent's parent (black node) make up 3 values. This is a full node! 
        - Therefore, there is no more space for a new insertion, and we need to trigger a **node split** 
    - How do we perform a node split in RBT?
        - We flip the colour of the parent and uncle from red to black, signifying that they become standalone nodes themselves
        - We flip the parent's parent from black to red, and see if the parent can be added to an existing node up the chain
        - Move your pointer up to the Grandparent. Treat it as a newly inserted red node, and check if *it* now causes a double-red violation with its own parent.

        ```
                G (Black)                          G (Red)  <-- Check up here next!
                /         \                        /       \
            P (Red)     U (Red)      --->      P (Black)  U (Black)
            /                                  /
        X (Red)                             X (Red)
        ```

- Case 2: The Uncle is BLACK or NIL (Rotations + Recoloring)
    - This scenario means the virtual B-Tree node is not full, but our binary tree presentation is physically lopsided and misaligned. 
    - We now use rotations to restructure the nodes into a uniform, balanced sub-component.

    - Sub-Case 2A: Linear Alignment (Left-Left or Right-Right)
        - The path from the Grandparent to the newly inserted node forms a straight line.

        - Solution:
            - Perform a **Single Rotation** around the Grandparent (Right-rotate if Left-Left; Left-rotate if Right-Right).
            - Swap the colors of the **original Parent** and the **original Grandparent** (Parent becomes Black, Grandparent becomes Red).
            - **The Result:** Balance is perfectly restored, and the loop terminates immediately.

            ```
                    G (Black)                       P (Black)
                    /         \                     /         \
                P (Red)     U (Black)    --->    X (Red)     G (Red)
                /                                               \
            X (Red)                                            U (Black)
            ```

    - Sub-Case 2B: Zig-Zag Alignment (Left-Right or Right-Left)
        - The path from the Grandparent to the newly inserted node forms a jagged zig-zag line.

        - Solution:
            - This cannot be fixed in a single step. We must first convert the zig-zag into a straight line.
            - Perform a **Single Rotation around the Parent** (Left-rotate if Left-Right; Right-rotate if Right-Left).
            - The tree is now arranged in a straight line (Sub-Case 2A). Treat your old Parent as the new inserted node and apply the standard 2A rotation/recoloring.

        ```
            G (Black)                     G (Black)                  X (Black)
            /         \                   /         \                /         \
        P (Red)     U (Black)  --->   X (Red)     U (Black)  ---> P (Red)     G (Red)
            \                           /                                             \
            X (Red)                  P (Red)                                         U (Black)
        ```

#### Deletion

- When a node is deleted from a Red-Black Tree, we begin with a standard Binary Search Tree (BST) deletion. If the node physically removed (or shifted) has an original color of RED, the tree's balance is completely unaffected. 

- But if the removed node's original color is **BLACK**, now we have a problem:
    - We will have 100% broken Rule 5 (Perfect Black Balance). One specific path now has fewer black nodes than all the other paths.
    - To track this structural deficit, we conceptually pass the "missing black point" to the node that takes the place of the deleted node. This node becomes Double-Black (if it was Black) or Black (if it was Red).
        - That is, we just find some node, and either make it count as a new black node (if it were red),  or count as 2 black nodes (if it were black)
        - Then, we "pass" the extra black node up the tree until we find a red node we can turn black, thereby restoring tree balance

- This section will look at how we can pass this black node back up. There are 4 cases to deal with:

##### Case 1: Sibling is RED

- The Double-Black node X has a Red sibling. This means the virtual B-Tree nodes are misaligned across layers.

- The Strategy: Transform the tree structure so that X gets a Black sibling, allowing us to use the other cases.

- The Fix:
    - Switch the colors of the Parent (make it RED) and the Sibling (make it BLACK).
    - Perform a Left Rotation around the Parent (if X is a left child).
    - Update the sibling pointer (the new sibling becomes the old sibling's left child). 

- **NOTE:** By itself, this fixes nothing. The point of this is to give us a scenario where the double black node has a black sibling, which is necessary for cases 2, 3, and 4

- Next Step: You are now safely inside Case 2, 3, or 4. Continue matching within the loop.
    ```
        P (Black)                      S (Black)
        /         \                    /         \
    X (DB)       S (Red)    --->    P (Red)       B
                /     \             /     \
               A       B        X (DB)     A
    ```

##### Case 2: Sibling is BLACK, and Both of Sibling's Children are BLACK

- This scenario represents an underflow inside the local virtual B-Tree node layer
    - The Strategy: Pull a layer of blackness out of both branches of the parent to balance them locally, then pass the deficit upward.
    - The Fix:
        - Change the Sibling color to RED (stripping its black point).
        - Move the Double-Black pointer up to the Parent (X = X.parent).
    - The Loop: If the parent was Red, it becomes a normal Black node, and balance is perfectly restored (the loop terminates). If the parent was Black, it is now the new Double-Black node, and the loop repeats up the chain.
        
        ```
            P (Red/Black)                   P (Black/Double Black)
            /         \                    /               \
        X (DB)       S (Black)  --->     X (Black)        S (Red)
                    /         \                           /     \
                Left(B)      Right(B)                 Left(B)  Right(B)
        ```

##### Case 3: Sibling is BLACK, Sibling's Inner Child is RED, Outer Child is BLACK

- The path from the parent to the sibling's red child forms a jagged zig-zag line (e.g., X is a left child, S is a right child, but S.left is Red).

- The Strategy: This cannot be resolved in one step. We convert the zig-zag alignment into a straight line to fit Case 4.
    - The Fix:
        - Swap the colors of the Sibling (becomes RED) and its Inner Red Child (becomes BLACK)
        - Perform a Right Rotation around the Sibling.
        - Update the sibling pointer (the inner child is now the new sibling).

- Next Step: The tree is now in a straight line. Proceed directly into Case 4.

- Diagram
    - Step 1
        ```
            P (Red/Black)
            /             \
        X (DB)           S (Black)
                        /         \
                    Inner (Red)   Outer (Black)
        ```

    - Step 2: Before moving any pointers, the code swaps the colors of S and Inner.
        ```
            P (Red/Black)
            /             \
        X (DB)           S (Red)         <-- Changed to RED
                        /         \
                    Inner (Black)   Outer (Black)  <-- Changed to BLACK
        ```
    
    - Step 3: perform a standard BST Right Rotation using S as the root of the rotation subtree
        ```
              P (Red/Black)
            /             \
        X (DB)           Inner (Black)
                                \
                                S (Red)
                                \
                                    Outer (Black)
        ```

##### Case 4: Sibling is BLACK, Sibling's Outer Child is RED

- The path from the parent to the sibling's red child forms a perfectly straight line (e.g., X is a left child, S is a right child, and S.right is Red). This represents an abundance of keys in the neighboring virtual B-Tree node, allowing us to borrow a key to fill our deficit.

- The Fix:
    1. Give the Sibling the exact same color as the Parent.
    2. Change the Parent color to BLACK.
    3. Change the Sibling's Outer Child color to BLACK.
    4. Perform a Left Rotation around the Parent.

- The Result: The structural deficit is completely absorbed. Set X = self.root to instantly terminate the rebalancing loop.

- Diagram
    - Step 1: 
        ```
            P (P_Color)
            /           \
        X (DB)         S (Black)
                    /         \
                Left (Black)   Outer (RED)  🔥
        ```
    
    - Step 2: Before moving any pointers, the code performs a three-way recoloring to prepare the nodes for their new structural roles:
        ```
              P (BLACK)        <-- Changed to BLACK
            /           \
        X (DB)         S (P_Color)    <-- Takes P's original color
                        /         \
                    Left (Black)   Outer (BLACK) <-- Changed to BLACK
        ```

    - Step 3: Now, the code executes a Left Rotation around $P$. 
        ```
                    S (P_Color)
                /           \
            P (BLACK)       Outer (BLACK)
            /         \
        X (DB)       Left (BLACK)
        ```


##### Key Takeaways

1. Max Rotations: Unlike an AVL tree deletion which can trigger a cascade of structural changes all the way up the branch (O(log n) rotations), a Red-Black Tree deletion will execute at most 3 rotations total before the structural deficit is completely neutralized and the balance is restored.

2. The Mirror Property: The code you see in _fix_delete uses the exact mirror image of this logic when the Double-Black node X happens to be a right child instead of a left child (swapping left/right pointers and left/right rotation triggers).

### Time complexity of Red-Black Trees

| Operation | Average Time | Worst Time | Space Overhead | Why? |
| :--- | :--- | :--- | :--- | :--- |
| **Search** | O(log n) | O(log n) | O(1) | Guaranteed strict logarithmic bound due to perfect Black Height balancing. |
| **Insertion** | O(log n) | O(log n) | O(1) | Finding the location takes O(log n). Rebalancing requires at most O(log n) color flips up the tree, but at most **2 structural rotations** total. |
| **Deletion** | O(log n) | O(log n) | O(1) | Finding the node takes O(log n). Rebalancing against a double-black path deficit requires at most **3 structural rotations**. |

### Parallel of RBT operations with B Trees

- Insertion Recoloring = A B-Tree Node Split
    - Think back to Insertion Case 1: You insert a new Red node (`X`), but its parent (`P`) is Red and its uncle (`U`) is Red.
        ```
                G (Black)   <-- The Grandparent Anchor
                /         \
            P (Red)     U (Red)
            /
        X (Red)  <-- Newly Inserted Element
        ```

    - Look at that cluster. You have a Black anchor (`G`) and two Red co-inhabitants (`P` and `U`), and you are trying to wedge a fourth item (`X`) into that exact same space.

    - Count the keys: G + P + U + X = 4 keys.

    - A 2-3-4 B-Tree node can only hold a maximum of 3 keys. Attempting to fit 4 keys means the B-Tree node has overflowed and must split!

    - When a B-Tree node splits, the middle key gets promoted up to the parent layer, while the remaining keys split into two separate, independent smaller nodes.

    - Watch the RBT recoloring step do exactly this:
        ```
                G (Red)     <-- PROMOTED up to the higher B-Tree layer!
                /       \
            P (Black)  U (Black) <-- Split into two independent B-Tree nodes
            /
        X (Red)
        ```

    - By turning `P` and `U` Black, the RBT says: "You two are no longer co-inhabitants of G's node. You are now independent, standalone B-Tree node anchors." By turning `G` Red, it passes `G` up to the higher level to see if it can fit into the parent layer above.

- Deletion Case 2 = Merging Underflowed Nodes
    - When we delete a Black node, we leave a structural deficit ("Double-Black" status). In a B-Tree, deleting a key causes an underflow (the node drops below the minimum allowed number of keys). 
    
    - If a node underflows, and its sibling next door is also poor, they cannot trade keys. They must merge together.

        ```
            P (Red or Black)
            /               \
        X (DB)             S (Black)
                        /         \
                    Left(B)      Right(B)
        ```

    - In B-Tree language, `S` being Black with two Black children means `S` is living  completely alone in its virtual B-Tree node. It has no extra keys to spare. It is "poor."

    - Because the left branch (`X`) has a deficit and the right branch (`S`) has nothing to give, our only choice is to merge layers by pulling a layer of blackness out of both branches to balance them locally.

    - The RBT Execution:
        ```
            P (Black / Double-Black) <-- Deficit passed upward
            /               \
        X (Black)          S (Red)   <-- Collapsed to merge with P
                            /       \
                        Left(B)   Right(B)
        ```

    - By turning `S` from Black to Red, the RBT is physically dissolving a node boundary. It collapses `S` so that it can merge back into `P`, while the localized deficit is passed up to `P` to be dealt with at a higher level.    

- Deletion Case 4 = Borrowing from a Wealthy Sibling
    - If a node underflows, but its sibling next door has a Red child, it means the sibling node contains multiple keys. It is "wealthy" and has a spare key to give!
        ```
            P (Black)
            /         \
        X (DB)       S (Black)
                        \
                            Outer (Red)  <-- Wealthy extra key!
        ```

    - Because the right side has an extra key, we perform a B-Tree rotation: we pull the parent key `P` down to fill our deficit, and we pass the sibling's extra key `Outer` up to replace the parent.

    - The RBT Execution: To make this switch happen, the RBT dynamically changes the colors right before it shifts the pointers via rotation:
        ```
        Step A: Color Shift                     Step B: Left Rotate around P
        ---------------------                   ------------------------------
            P (BLACK)                                   S (P_Color)
            /         \                                 /           \
        X (DB)       S (P_Color)                    P (BLACK)      Outer (BLACK)
                            \                        /      \
                        Outer (BLACK)         X (Black)    Left (Black)
        ```

    - Look at the final result of Step B:
        - `Outer` turns from Red to Black, becoming its own independent, structural node anchor.
        - `P` turns Black directly above `X`. This extra injection of blackness instantly absorbs `X`'s double-black status, downgrading it back to a normal, happy single Black node.

### Implementation

In [ ]:
class Node:
    def __init__(self, data, color="RED"):
        self.data = data
        self.left = None
        self.right = None
        self.parent = None
        
        # 1 extra bit of information to hold "RED" or "BLACK" info. Everything above is the
        # same as a BST node
        self.color = color 

class RedBlackTree:
    def __init__(self):
        # The universal Black NIL Sentinel Node
        self.NIL = Node(data=None, color="BLACK")
        self.root = self.NIL

    def left_rotate(self, x):
        y = x.right
        x.right = y.left
        if y.left != self.NIL:
            y.left.parent = x
        y.parent = x.parent
        if x.parent is None:
            self.root = y
        elif x == x.parent.left:
            x.parent.left = y
        else:
            x.parent.right = y
        y.left = x
        x.parent = y

    def right_rotate(self, y):
        x = y.left
        y.left = x.right
        if x.right != self.NIL:
            x.right.parent = y
        x.parent = y.parent
        if y.parent is None:
            self.root = x
        elif y == y.parent.right:
            y.parent.right = x
        else:
            y.parent.left = x
        x.right = y
        y.parent = x

    def insert(self, data):
        new_node = Node(data, color="RED")
        new_node.left = self.NIL
        new_node.right = self.NIL
        
        y = None
        x = self.root
        
        while x != self.NIL:
            y = x
            if new_node.data < x.data:
                x = x.left
            else:
                x = x.right
                
        new_node.parent = y
        
        if y is None:
            self.root = new_node
        elif new_node.data < y.data:
            y.left = new_node
        else:
            y.right = new_node
            
        if new_node.parent is None:
            new_node.color = "BLACK"
            return
            
        if new_node.parent.parent is None:
            return
            
        self._fix_insert(new_node)

    def _fix_insert(self, k):
        while k.parent.color == "RED":
            if k.parent == k.parent.parent.left:
                uncle = k.parent.parent.right
                if uncle.color == "RED":
                    k.parent.color = "BLACK"
                    uncle.color = "BLACK"
                    k.parent.parent.color = "RED"
                    k = k.parent.parent 
                else:
                    if k == k.parent.right:
                        k = k.parent
                        self.left_rotate(k)
                    k.parent.color = "BLACK"
                    k.parent.parent.color = "RED"
                    self.right_rotate(k.parent.parent)
            else:
                uncle = k.parent.parent.left
                if uncle.color == "RED":
                    k.parent.color = "BLACK"
                    uncle.color = "BLACK"
                    k.parent.parent.color = "RED"
                    k = k.parent.parent
                else:
                    if k == k.parent.left:
                        k = k.parent
                        self.right_rotate(k)
                    k.parent.color = "BLACK"
                    k.parent.parent.color = "RED"
                    self.left_rotate(k.parent.parent)
            if k == self.root:
                break
        self.root.color = "BLACK"

    def _transplant(self, u, v):
        """Helper to swap subtrees during standard BST deletion"""
        if u.parent is None:
            self.root = v
        elif u == u.parent.left:
            u.parent.left = v
        else:
            u.parent.right = v
        v.parent = u.parent

    def _minimum(self, node):
        """Helper to find the successor node"""
        while node.left != self.NIL:
            node = node.left
        return node
    
    def delete(self, data):
        # Locate the node to delete
        z = self.root
        while z != self.NIL and z.data != data:
            if data < z.data:
                z = z.left
            else:
                z = z.right

        if z == self.NIL:
            print(f"Value {data} not found in the tree.")
            return

        y = z
        y_original_color = y.color
        
        if z.left == self.NIL:
            x = z.right
            self._transplant(z, z.right)
            # Fix: Ensure NIL tracks its structural parent safely if passed to fix_delete
            if x == self.NIL:
                self.NIL.parent = z.parent
                
        elif z.right == self.NIL:
            x = z.left
            self._transplant(z, z.left)
            # Fix: Ensure NIL tracks its structural parent safely if passed to fix_delete
            if x == self.NIL:
                self.NIL.parent = z.parent
                
        else:
            y = self._minimum(z.right)
            y_original_color = y.color
            x = y.right
            
            if y.parent == z:
                # Fix: Don't overwrite global NIL parent pointer if x is NIL
                if x != self.NIL:
                    x.parent = y
                else:
                    self.NIL.parent = y
            else:
                self._transplant(y, y.right)
                y.right = z.right
                y.right.parent = y
                
            self._transplant(z, y)
            y.left = z.left
            y.left.parent = y
            y.color = z.color

        if y_original_color == "BLACK":
            self._fix_delete(x)

    def _fix_delete(self, x):
        while x != self.root and x.color == "BLACK":
            if x == x.parent.left:
                s = x.parent.right
                # Case 1: Sibling is RED
                if s.color == "RED":
                    s.color = "BLACK"
                    x.parent.color = "RED"
                    self.left_rotate(x.parent)
                    s = x.parent.right

                # Case 2: Sibling is BLACK, both children are BLACK
                if s.left.color == "BLACK" and s.right.color == "BLACK":
                    s.color = "RED"
                    x = x.parent
                else:
                    # Case 3: Sibling is BLACK, inner child is RED, outer is BLACK
                    if s.right.color == "BLACK":
                        s.left.color = "BLACK"
                        s.color = "RED"
                        self.right_rotate(s)
                        s = x.parent.right

                    # Case 4: Sibling is BLACK, outer child is RED
                    s.color = x.parent.color
                    x.parent.color = "BLACK"
                    s.right.color = "BLACK"
                    self.left_rotate(x.parent)
                    x = self.root
            else:
                # Mirror cases (when X is a right child)
                s = x.parent.left
                if s.color == "RED":
                    s.color = "BLACK"
                    x.parent.color = "RED"
                    self.right_rotate(x.parent)
                    s = x.parent.left

                if s.right.color == "BLACK" and s.left.color == "BLACK":
                    s.color = "RED"
                    x = x.parent
                else:
                    if s.left.color == "BLACK":
                        s.right.color = "BLACK"
                        s.color = "RED"
                        self.left_rotate(s)
                        s = x.parent.left

                    s.color = x.parent.color
                    x.parent.color = "BLACK"
                    s.left.color = "BLACK"
                    self.right_rotate(x.parent)
                    x = self.root
        x.color = "BLACK"
        # Reset NIL parent clean-up to prevent side effects in future insertions/deletions
        self.NIL.parent = None

Inorder Traversal of Balanced RBT:
10(BLACK) 20(BLACK) 30(RED) 40(BLACK) 50(RED) 
Root of the tree is: 20 (BLACK)
